# YuE2 on Kaggle — restored native FP16 engine (revision 12)

**First time:** run `yue2-build-kaggle-assets.ipynb` once on Kaggle with
Internet and **no GPU**, then save its outputs as a private dataset.
Add that dataset to this notebook with **Add Input**. You can also attach
the saved builder notebook's output directly. Nothing large needs to be
downloaded to your PC or uploaded from your PC.

**Every music session:** attach that saved input, select **GPU T4 ×2**
and enable Internet for the Gradio and optional Cloudflare links. Setup reads weights from
`/kaggle/input` and installs dependencies from saved local wheels.
Missing/incompatible assets cause an early error, never a replacement
download. Keep the same Kaggle Python/image version as the builder;
upgrades can require rebuilding the dependency bundle.

This restores the original
`yue2_infer 0.1.5` engine: FP16 transformer on GPU 0, CUDA-graph AR decoding
with SDPA, and tiled FP32 VAE on GPU 1. Default generation uses **Direct to
audio**, **16 Euler steps**, CFG **1.0**, and an **8-minute safety ceiling**.
The model chooses a natural ending from your prompt and lyrics. It does
not have to fill the ceiling. A token-limit cutoff is reported separately.

**No diagnostic song runs automatically.** Run setup and open the UI,
enter your full song lyrics and style, leave or adjust the safety ceiling, then click
Generate Song. Optional score planning remains off unless you select it.
**Cover your own song:** open **Cover / restyle my existing song** in the
UI, upload its WAV/FLAC/MP3, paste your original lyrics, and click
**Analyze melody for cover**. The score fills Custom ABC and selects
Melody score mode. Enter the new style, then Generate Song. Optional
**Transcribe lyrics too** uses Whisper when needed. Extracted-score
regeneration can change timing, voice and arrangement; it is not exact
audio-to-audio copying. Melody analysis reads SheetSage2 and its MERT
parent from the saved dataset; optional Whisper is included there too. Download the
cover-analysis ZIP to retain the editable score. Transcription models
are unloaded after analysis and GPU jobs share one queue.

**T4 attention change:** equivalent grouped-head views make PyTorch's
memory-efficient kernel eligible, instead of the native T4 GQA math path.
The first supported CUDA dispatch is verified and logged. Unsupported
shapes retain native attention and are reported. CPU equivalence checks
passed; actual T4 speed improvement has not been measured.

Durations are maximums. A natural early ending or token cutoff is reported.
Long acoustic generations may span multiple chunks; listen at the reported
boundaries. Seven/eight-minute performance and continuity remain unverified.

The small installed environment uses session-only `/kaggle/temp/yue2_native`.
Large models stay in the attached read-only Kaggle input dataset.
Song audio, exact tokens and recovery latents use
`/kaggle/working/yue2_native/runs`. The paths may share underlying storage;
setup prints actual free space rather than assuming a separate 50 GB quota.
Old notebook caches and files are preserved.

**Process protection:** setup and launch stop only verified older engines
using this script/output root, including survivors of a kernel restart.
A process lock prevents two managed engines from loading together. GPU
memory is checked before loading and before generation. An unrecognized
GPU process is reported and left running; setup stops before generation.
CUDA OOM saves GPU process details and releases the failed engine's model.

**After generation, download the complete run ZIP in the UI, or Save
Version with outputs in Kaggle before ending the session.** A saved draft
or a notebook download containing logs does not preserve audio files.


**Start here:** attach `yue2-music-assets`, select GPU T4 x2 and Internet ON,
then run the next code cell. It performs setup and opens the generator in
one operation. Keep it running while using either UI link. Nothing
generates until you press Generate in the UI.

Setup/engine work runs in separate processes. A kernel restart does not
require reusing variables from another cell. Full startup details are saved
to `/kaggle/working/yue2_native/runtime/startup.log`; `startup-status.json`
records setup, engine startup, ready or failure. The independent file-list
cell below starts no setup or GPU job.


In [ ]:
import base64, json, os, signal, subprocess, sys, zlib
from pathlib import Path
SUPPORT = Path("/kaggle/temp/yue2_native_support")
ROOT = "/kaggle/temp/yue2_native"
OUTPUTS = "/kaggle/working/yue2_native/runs"
ASSET_DATASET = "auto"
ENABLE_CLOUDFLARE = True
SUPPORT.mkdir(parents=True, exist_ok=True)
payload = "..."
for relative, contents in json.loads(zlib.decompress(base64.b64decode(payload))).items():
    path = SUPPORT / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
env = dict(os.environ)
env["PYTHONPATH"] = str(SUPPORT) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTHONUNBUFFERED"] = "1"
arguments = [sys.executable, "-u", "-m", "native_yue2.launch",
             "--root", ROOT, "--outputs", OUTPUTS, "--assets", ASSET_DATASET]
if not ENABLE_CLOUDFLARE:
    arguments.append("--no-cloudflare")
if globals().get("RECOVER_RUN"):
    arguments.extend(["--recover", str(RECOVER_RUN)])
log_path = Path(OUTPUTS).parent / "runtime/startup.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
worker = subprocess.Popen(arguments, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, start_new_session=True)
try:
    with log_path.open("a") as log:
        for line in worker.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
    code = worker.wait()
    if code:
        raise RuntimeError(f"Startup stopped (exit {code}). Full details: {log_path}")
finally:
    if worker.poll() is None:
        os.killpg(worker.pid, signal.SIGTERM)
        try:
            worker.wait(timeout=20)
        except subprocess.TimeoutExpired:
            os.killpg(worker.pid, signal.SIGKILL)
            worker.wait(timeout=5)


**Find and preserve your song.** Each attempt has its own folder. After a
successful generation the UI offers WAV, FLAC, and a complete ZIP containing
settings, tokens, timings and recovery latents. Download it before stopping.
The folder also appears under Kaggle Output at `yue2_native/runs`.
`status.json` records the measured audio length and any cutoff; it does not
establish musical completeness. Listen to the ending and chunk boundaries.

Run the next cell after stopping the UI to list all saved attempts. It
starts no model or generation. Files still need to be downloaded or saved
with a Kaggle output version before the session is discarded.


In [ ]:
from pathlib import Path
import json
OUTPUTS = "/kaggle/working/yue2_native/runs"
from IPython.display import FileLink, display
for directory in sorted(Path(OUTPUTS).glob("*")):
    if directory.is_dir():
        status = directory / "status.json"
        print(directory.name, status.read_text() if status.exists() else "Partial run")
        for filename in ("audio.wav", "audio.flac"):
            if (directory / filename).exists():
                display(FileLink(str(directory / filename)))
for archive in sorted(Path(OUTPUTS).glob("*.zip")):
    display(FileLink(str(archive)))

if not Path(OUTPUTS).exists():
    print("No saved song folders in this session yet.")


Optional recovery: set RECOVER_RUN below, then rerun the startup cell. Leave None for normal music creation.


In [ ]:
# Optional: only after stopping the generator.
RECOVER_RUN = None
if RECOVER_RUN:
    print("Recovery selected. Run the startup cell above; it will decode these saved latents.")


Based on the original AIQUEST Academy notebook in Movies commit
`fcf4d45dc5dbca57b39e54311807410cdf41f737`.
Sources: [YuE2-3B](https://huggingface.co/m-a-p/YuE2-3B),
[YuE2 VAE](https://huggingface.co/m-a-p/YuE2-Vae).
The native inference wheel and model revisions are pinned; wheel and
model checksums are verified. Upstream model/license terms apply.
Source and checks are included in the Music repository for review.
